# Module 2 — Text Preprocessing

**What you are doing today:** taking the messy messages you collected and
turning them into something a computer can actually match on.

And then finding out that cleaning too hard destroys the meaning.

| Part | What you do |
| --- | --- |
| **A — Follow along** | Run each cell, read what to notice. Nothing to write. |
| **B — Your turn** | Six tasks. Now you write the code. |

**Have your 10 collected messages open.** You need them in task B6.

Work in pairs. Swap who types between the two parts.


---
# Part A — Follow along

Seven short steps, A1 to A7, after one setup cell. **Run each cell, then read the
*Notice* line under it.** You do not write anything in this part.

You are building one function, `clean_text()`, one stage at a time. By A6 it is
finished; A7 adds the piece that fixes last module's typos.


### Setup — run this first

Not a step, just the plumbing. It fetches a list of English filler words that
somebody else has already written down for us.

If the download is blocked on your school network, the cell quietly falls back to
a short built-in list and everything still works.

In [ ]:
try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
    print("NLTK list loaded:", len(STOP), "words")
except Exception:
    STOP = set("i me my we you he she it they a an the is am are was were be "
               "been do does did have has had of to in on at for with and or "
               "but if this that these those not no can will just".split())
    print("Download blocked — using the built-in list:", len(STOP), "words")


**Notice:** the real list has **198** words. Nobody at your school chose them —
someone made that list years ago and every project since has just used it. Hold
on to that thought; it comes back in A5.

### A1 — The message that beat your bot

Here is your bot from last module, and one real customer message.

In [ ]:
def bot(msg):
    if "hello" in msg:
        return "Hi! How can I help?"
    elif "order" in msg:
        return "Your order is on the way."
    elif "bye" in msg:
        return "Goodbye!"
    else:
        return "Sorry, I didn't catch that."


message = "Helo, WHERE IS MY ORDER?!"
print(bot(message))


**Notice:** the customer said the word ORDER, in capitals, right there in the
message — and the bot still missed it. Nothing is wrong with the customer.

Everything in Part A is about closing that gap.

### A2 — Stage 1: lowercase

`.lower()` makes every letter small.

In [ ]:
message = "Helo, WHERE IS MY ORDER?!"

lowered = message.lower()
print(lowered)
print(bot(lowered))


**Notice:** one method call and the bot now answers correctly.

`ORDER` and `order` are different words to a computer. They are not different
words to a customer. Lowercasing is how you settle that argument.

### A3 — Stage 2: remove punctuation

`?!,.` are not part of any word, but `"order"` does not match `"order?!"`.

In [ ]:
import string

print(string.punctuation)

no_punct = lowered.translate(str.maketrans("", "", string.punctuation))
print(no_punct)


**Notice:** `string.punctuation` is just a string of 32 characters, and
`str.maketrans("", "", string.punctuation)` says *delete every one of these*.

You do not need to remember that line. You do need to remember **why** it is
there: `order?` and `order` must become the same thing.

### A4 — Stage 3: split into words

One long string is hard to work with. A list of words is easy.

In [ ]:
words = no_punct.split()
print(words)
print(len(words), "words")


**Notice:** `.split()` cuts on spaces. That is all it does, and it is not
enough — but everything it gets wrong needs its own lesson, so for now:

| Splitting on spaces gets this wrong | Which is it? |
| --- | --- |
| `don't` | one word or two? |
| `New York` | two words or one? |
| `RM1,200` | you just deleted the comma in A3 |
| `order#4521` | you just deleted the `#` too |

Nobody has a clean answer. Real systems all pick a compromise and live with it.

These pieces have a name: **tokens.** Cutting text into them is **tokenisation**.

### A5 — Stage 4: drop the filler words

Words like *is*, *my* and *the* appear in almost every message, so they tell you
almost nothing about what the customer wants. Dropping them makes matching
faster and the message shorter.

In [ ]:
kept = [w for w in words if w not in STOP]
print(kept)

# Now the same four stages on a different message.
message2 = "I do NOT want a refund"
w2 = message2.lower().translate(str.maketrans("", "", string.punctuation)).split()
print([w for w in w2 if w not in STOP])


**Notice the second one.**

`I do NOT want a refund` became **`['want', 'refund']`**.

The customer said they did **not** want a refund. Your cleaned version says they
do. You have not lost some detail — you have reversed the meaning.

`not` is in the stopword list. So is `no`. **`never` is not** — run
`print("never" in STOP)` and see. Nobody designed that inconsistency; it is just
what is in the file everyone downloads.

This is the most important cell in the module. Read it twice.

### A6 — All four stages in one function

Nothing new here — the same four lines, given a name.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [w for w in words if w not in STOP]
    return words


print(clean_text("Helo, WHERE IS MY ORDER?!"))
print(clean_text("i wan 2 chk my ordr"))


**Notice the second line of output:** `['wan', '2', 'chk', 'ordr']`.

Perfectly cleaned. Lowercased, punctuation gone, filler gone — and still not one
word your bot is looking for. Cleaning cannot fix spelling.

That is A7.

### A7 — Fixing typos by closeness

How many single-letter changes turn `ordr` into `order`? One — insert an `e`.
And `oder`? Also one. That count is called **edit distance**, and it is the whole
idea behind spellcheck.

Python has it built in. Nothing to install.

In [ ]:
from difflib import get_close_matches

KEYWORDS = ["order", "parcel", "delivery", "refund", "return", "hello", "bye"]


def fix_typos(words, cutoff=0.7):
    fixed = []
    for w in words:
        match = get_close_matches(w, KEYWORDS, n=1, cutoff=cutoff)
        fixed.append(match[0] if match else w)
    return fixed


for m in ["i wan 2 chk my ordr", "Helo anyone there", "wheres my parcl"]:
    print(clean_text(m), "->", fix_typos(clean_text(m)))


**Notice:** `ordr` → `order`, `helo` → `hello`, `parcl` → `parcel`. The bot can
finally see the words it was looking for.

`cutoff=0.7` is the strictness dial: *how similar is close enough?* Somebody
chose 0.7. In B5 you find out what happens when you choose differently.

**Part A is done.** You now have `clean_text()` and `fix_typos()`. Part B is
yours.

---
# Part B — Your turn

Six tasks. You are expected to get things wrong here — that is what the part is
for. Nothing in Part B is marked.

If a cell will not run, read the **last** line of the red error first. It is
usually the useful one.

### B1 — How often does that happen?

In A5 you watched `I do not want a refund` turn into `want refund`.

One sentence is an anecdote. Below are **15 real customer messages**. Run them all
through `clean_text()` and find out how big the problem actually is.

**Your job: find every message where the cleaned version no longer means what the
customer meant.** Not "looks shorter" — *means something different.*

Several messages lose half their words and still mean exactly the same thing.
Those do not count. There are more than two and fewer than six.

In [ ]:
messages = [
    "I do not want a refund",
    "WHERE IS MY ORDER???",
    "Helo, i wan chk my ordr",
    "Where is my parcel?",
    "hi, my order still havent arrive",
    "can u check my ordr status",
    "still waiting for my delivery leh",
    "my parcel not here yet",
    "tracking pls",
    "can i return the shoes i bought last week ah",
    "the item is not what i ordered",
    "i paid alredy but no email",
    "Is My ORDER Here? The Box Is Broken.",
    "no need already, cancel it",
    "thanks bye",
]

for m in messages:
    print(f"{m:46} ->  {' '.join(clean_text(m))}")


**Write down every message where the meaning changed, not just the length:**

_______________________________________________

_______________________________________________

_______________________________________________

_______________________________________________

**Then:** what do the ones you found have in common?

_______________________________________________

### B2 — Write your own filler list

The NLTK list has 198 words and you did not choose any of them. Now you choose.

Fill in `MY_STOP` with the filler words **you** think should be dropped from a
customer-service message. Somewhere between 8 and 20 words is sensible.

Then decide the question A5 raised: **is `not` going in your list or not?** You
have to pick one, and either answer costs you something.

In [ ]:
# Your filler list. Between 8 and 20 words.
MY_STOP = {
    "is", "my", "the", "___", "___", "___",
}


def my_clean(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [w for w in words if w not in ___]
    return words


# Test it on both of these before you move on.
print(my_clean("Where is my order?"))
print(my_clean("I do not want a refund"))


### B3 — Predict, then run

**Write your prediction first.** Double-click the table to type in it.

For each message: after `clean_text()` runs, will your Module 1 bot match it?

| Message | Your prediction | What actually happened |
| --- | --- | --- |
| `WHERE IS MY ORDER???` |  |  |
| `Helo, i wan chk my ordr` |  |  |
| `Where is my parcel?` |  |  |
| `I do not want a refund` |  |  |

Now run it.

In [ ]:
tests = [
    "WHERE IS MY ORDER???",
    "Helo, i wan chk my ordr",
    "Where is my parcel?",
    "I do not want a refund",
]

for t in tests:
    cleaned = fix_typos(clean_text(t))
    print(f"{t:30} -> {str(cleaned):45} -> {bot(' '.join(cleaned))}")


### B4 — Find the bug

This `clean_text_broken()` looks right and is not. It returns almost nothing
useful for the message below.

**Run it first. Then read it and work out why** — do not fix it by guessing.

In [ ]:
def clean_text_broken(text):
    words = text.split()
    words = [w for w in words if w not in STOP]
    text = " ".join(words)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()


print(clean_text_broken("Is My ORDER Here? The Box Is Broken."))
print(clean_text("Is My ORDER Here? The Box Is Broken."))


**Write your answer here.** Which stage is in the wrong place, and why does
that break it?

_______________________________________________

<br><br><br><br><br><br>

**The answer:** the filler words are removed **before** lowercasing. The list
contains `is`, `my` and `the` in small letters — so `Is`, `My` and `The` with
capitals never match it and survive.

In this sentence *every* word is capitalised, so the filler stage removed
**nothing at all** — and the output still looks perfectly clean, because the
later stages did their jobs. That is the dangerous kind of bug: no error, no
warning, just a quietly wrong answer.

Order matters. Lowercase first, always.

### B5 — Turn the strictness dial

`fix_typos` has a `cutoff`. Higher means stricter. Somebody chose `0.7` for you.

Run the cell as it is, then change `cutoff` to **0.9**, then to **0.4**, and
record what breaks at each end.

In [ ]:
messages = [
    "i wan 2 chk my ordr",
    "i paid alredy but no email",
    "the red one i orderd is broken",
    "helo anyone there",
]

CUTOFF = 0.7      # <-- change this to 0.9, then to 0.4

for m in messages:
    print(f"{m:32} -> {fix_typos(clean_text(m), cutoff=CUTOFF)}")


**Fill this in as you go:**

| Cutoff | What goes wrong |
| --- | --- |
| `0.9` |  |
| `0.7` |  |
| `0.4` |  |

**Then the question:** which cutoff would you ship, and what are you accepting
when you choose it?

_______________________________________________

### B6 — Score your own messages

Last module you collected 10 real customer messages and your bot failed most of
them. Now it has cleaning.

**Paste your 10 messages into the list below**, then run it. Two numbers come
out. Both go on the board.

In [ ]:
my_messages = [
    "Helo, i wan chk my ordr",
    "where is my parcel",
    "___",
    "___",
    "___",
    # ... all 10
]

my_messages = [m for m in my_messages if m != "___"]   # ignore any you left blank
FAIL = "Sorry, I didn't catch that."

before = sum(1 for m in my_messages if bot(m.lower()) != FAIL)
after = sum(1 for m in my_messages
            if bot(" ".join(fix_typos(clean_text(m)))) != FAIL)

print(f"Before cleaning: {before} / {len(my_messages)}")
print(f"After cleaning:  {after} / {len(my_messages)}")
print()
print("Still failing:")
for m in my_messages:
    if bot(" ".join(fix_typos(clean_text(m)))) == FAIL:
        print("  -", m)


**Now the part that matters.** Look at the messages that *still* fail.

Write down, for one of them, **why** cleaning could not save it:

_______________________________________________

_______________________________________________

---
## Stretch — word endings *(only if you finish early)*

`order`, `orders` and `ordered` are the same idea. To your matcher they are three
unrelated words.

Two tools claim to fix this, and they disagree about what "fix" means.

- A **stemmer** chops endings off by rule. Fast, crude, and it happily produces
  things that are not words.
- A **lemmatiser** looks the word up in a dictionary and returns the real base
  word. Slower, and it needs to be told what kind of word it is looking at.

Run it and watch them disagree.

In [ ]:
import nltk
nltk.download("wordnet", quiet=True)
from nltk.stem import PorterStemmer, WordNetLemmatizer

ps, wl = PorterStemmer(), WordNetLemmatizer()

for w in ["orders", "ordered", "delivering", "studies", "boxes", "ran", "better"]:
    print(f"{w:12} stem -> {ps.stem(w):10} lemma -> {wl.lemmatize(w):10} "
          f"lemma as a verb -> {wl.lemmatize(w, 'v')}")


**Two questions to answer.**

**1.** `delivering` stems to `deliv` and `studies` stems to `studi`. Neither is a
word. Is that a problem? *(Think about what the output is actually used for.)*

_______________________________________________

**2.** Look at `ran` and `better` in the middle column. The lemmatiser did
nothing at all — until it was told to treat the word as a verb. Why would a
program not already know that `ran` is a verb?

_______________________________________________

<br><br><br><br><br><br>

**Answers.**

**1.** No — as long as nothing ever shows it to a human. `studi` is a perfectly
good *key*: every form of the word lands on the same key, which is all the
matching needs. It becomes a problem the moment you print it on a screen.

**2.** Because working out a word's part of speech means looking at the whole
sentence, not the word. `ran` is a verb in *I ran home*; `better` is an adjective
in *a better price* and a verb in *she bettered her score*. That is the **syntax**
layer from Module 1 — the one the table said nothing fully fixes. The lemmatiser
simply assumes "noun" when nobody tells it, which is why it did nothing.


---
## Homework

Run **all 10** of your collected messages through `clean_text()` and `fix_typos()`.

For every one that **still** fails, write one sentence saying why.

Bring the list to the next module. We are going to fix most of them.

> **A tip that is also a warning:** "because the bot is stupid" is not a reason.
> "Because the bot looks for `order` and I wrote `package`" is.